[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/01-prerequisites/ml-prereq-sampling.ipynb)

# Sampling Methods & Design

*AIBits Academy · Machine Learning End To End · Prerequisites*

How you collect data determines what you're allowed to conclude from it — before any statistic gets computed, the sampling method has already set the ceiling on how trustworthy that statistic can be.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **📊 Why this page exists**
>
> Every other Prerequisites page assumes you already have a dataset in hand. This page is about the step before that: how a Zomato analyst decides which 5,000 orders to sample out of 5 million, and why that choice quietly determines whether the "average delivery time" you compute later actually means anything.

## Population vs. Sample

The **population** is the entire group you want to draw conclusions about (e.g. every Flipkart customer in India); a **sample** is the subset you actually observe and measure. Almost all of statistics exists because measuring the whole population is usually impossible or too expensive — everything from a confidence interval to a p-value is really a statement about how far a sample statistic is likely to be from the true population value.

## Probability Sampling: Every Unit Has a Known Chance of Selection

| Method | How it works | Example |
|---|---|---|
| Simple Random | Every member of the population has an equal chance of selection | Randomly select 1,000 HDFC Bank account holders from the full customer database |
| Stratified | Population split into subgroups (strata) by a known characteristic, then randomly sampled *within* each stratum | Sample proportionally from each of Surat, Ahmedabad, Mumbai, and Bengaluru so no city is over/under-represented |
| Cluster | Population split into naturally occurring clusters (e.g. stores, branches); a random *subset of whole clusters* is sampled, and every unit within a chosen cluster is included | Randomly pick 20 out of 200 Swiggy delivery hubs, then survey every rider at those 20 |
| Systematic | Pick a random starting point, then select every k-th unit from an ordered list | Every 50th transaction in Paytm's daily transaction log |

Gold/green dots = selected units. Simple Random scatters selections with no structure; Stratified spreads them evenly down two fixed columns (subgroups); Cluster picks two whole contiguous groups (neighbourhoods) and takes every unit inside them; Systematic ticks off every 4th unit in sequence.

## Non-Probability Sampling: Convenience Over Rigour

These methods don't give every population member a known (or equal) chance of selection — they're faster and cheaper, but the resulting sample can systematically misrepresent the population, and standard confidence intervals/p-values are not strictly valid on top of them.

> **Convenience Sampling**
>
> Sampling whoever is easiest to reach — e.g. surveying only customers who happen to visit a single Mehta Textiles showroom in Surat on a Tuesday. Fast, but likely biased toward that store's specific customer profile.

> **Quota Sampling**
>
> Fill fixed quotas per subgroup (e.g. "200 responses from each city") without randomizing *who* within that subgroup responds — resembles stratified sampling on the surface, but lacks the randomization that makes stratified sampling statistically valid.

> **⚠ Why this matters before you touch a model**
>
> A churn-prediction model trained only on customers who responded to a voluntary survey (a convenience sample) will learn the patterns of people willing to respond to surveys — not necessarily the patterns of churners in general. This is **selection bias**, and no amount of downstream modelling sophistication (Random Forest, XGBoost, or otherwise) can fix a sample that was biased at collection time.

## The Sampling Distribution of the Mean

If you drew many different samples of the same size from a population and computed the mean of each one, those sample means would themselves form a distribution — the **sampling distribution of the mean**. Its spread is measured by the **standard error**:

$$\text{Standard Error of the Mean:}\quad SE = \frac{\sigma}{\sqrt{n}}$$

Notice what this formula says: the standard error shrinks as sample size *n* grows, but only at a rate of √n — quadrupling your sample size only halves your standard error. This single relationship is why "just collect more data" has rapidly diminishing returns, and it sets up exactly the machinery the next page (Inferential Statistics) builds on for the Central Limit Theorem, confidence intervals, and hypothesis tests.

## Standard Error & Why Sample Size Matters

A sample mean is only an *estimate* of the population mean. The **standard error of the mean** (SE) measures how much that estimate would wobble from sample to sample: `SE = σ / √n`. Crucially, it shrinks with the *square root* of the sample size — to halve the error you need **four times** the data.

In [ ]:
import numpy as np

np.random.seed(1)
population = np.random.normal(50, 10, 1_000_000)
sigma = population.std()

for n in [10, 30, 100, 400]:
    se = sigma / np.sqrt(n)
    print(f"n={n:>3}: SE = {se:.3f}")

Diminishing returns: each extra data point helps less than the last.

> **💡 The link to the CLT**
>
> The Central Limit Theorem (see Inferential Statistics) says the sampling distribution of the mean is approximately normal with spread equal to this standard error — which is what makes confidence intervals and hypothesis tests possible.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Standard error by formula

Lumina Health's lab times have a population standard deviation of 12 minutes. Store the standard error of the mean for a sample of 36 patients in `se`.

In [ ]:
import numpy as np
sigma, n = 12, 36
se = None   # TODO


In [ ]:
try:
    check("se = 2.0", se == 2.0)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
sigma, n = 12, 36
se = sigma / np.sqrt(n)

```

</details>

### Exercise 2 · Medium · How many people do I need to survey?

Meridian Bank wants the standard error of average spend to be at most 1 (currency unit) when σ = 15. Store the smallest whole sample size in `n_needed`.

In [ ]:
import numpy as np
sigma, target_se = 15, 1
n_needed = None   # TODO


In [ ]:
try:
    check("225 respondents", n_needed == 225)
    check("meets the target", sigma / np.sqrt(n_needed) <= target_se)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
sigma, target_se = 15, 1
n_needed = int(np.ceil((sigma / target_se) ** 2))

```

SE = σ/√n, so n = (σ/SE)². Halving the error costs four times the sample.

</details>

### Exercise 3 · Stretch · Check the theory by simulation

Draw 2,000 samples of size 50 from a **skewed** population (`rng.exponential(scale=10, size=...)`). Store the standard deviation of the 2,000 sample means in `emp_se` and the theoretical value `sigma/sqrt(n)` (σ = 10 for this population) in `theory_se`.

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
n = 50
emp_se = theory_se = None   # TODO


In [ ]:
try:
    check("simulation agrees with theory within 10%", abs(emp_se - theory_se) / theory_se < 0.10)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
rng = np.random.default_rng(1)
n = 50
means = rng.exponential(scale=10, size=(2000, n)).mean(axis=1)
emp_se = means.std()
theory_se = 10 / np.sqrt(n)

```

Even though the population is far from bell-shaped, the sample means behave as the central limit theorem promises.

</details>

---
*Back to the course: **Machine Learning End To End → Sampling Methods & Design**.*